# Clase 8 — Visión por computador real: perros vs. gatos

**Objetivos de hoy**
- Preparar un dataset real de imágenes para entrenar una CNN.
- Aplicar aumento de datos para reducir el sobreajuste.
- Entrenar, evaluar e interpretar las predicciones de una CNN sobre fotos reales.

Consulta el [capítulo 7 del manual](../manual/07_dogs_vs_cats.md) para la explicación completa.

**Nota**: en lugar de descargar el dataset de Kaggle a mano (requiere credenciales), usamos aquí `tensorflow_datasets`, que incluye el mismo dataset "cats_vs_dogs" listo para cargar. La lógica de preprocesamiento, aumento de datos y arquitectura es idéntica a la que usarías con tus propias fotos descargadas de Kaggle, tal como se explica en el manual.

In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds

TAMANO_IMAGEN = (128, 128)
TAMANO_LOTE = 32

(dataset_entreno, dataset_validacion), info = tfds.load(
    "cats_vs_dogs",
    split=["train[:80%]", "train[80%:]"],
    as_supervised=True,
    with_info=True,
)

print("Número de clases:", info.features["label"].num_classes)

## 1. Preprocesa: mismo tamaño y píxeles normalizados para todas las imágenes

In [ ]:
def preprocesar(imagen, etiqueta):
    imagen = tf.image.resize(imagen, TAMANO_IMAGEN)
    imagen = imagen / 255.0
    return imagen, etiqueta

dataset_entreno = dataset_entreno.map(preprocesar).batch(TAMANO_LOTE).prefetch(1)
dataset_validacion = dataset_validacion.map(preprocesar).batch(TAMANO_LOTE).prefetch(1)

## 2. Proyecta unas cuantas imágenes de ejemplo

In [ ]:
import matplotlib.pyplot as plt

nombres_clase = info.features["label"].names
lote_imagenes, lote_etiquetas = next(iter(dataset_entreno))

plt.figure(figsize=(10, 4))
for i in range(6):
    plt.subplot(2, 3, i + 1)
    plt.imshow(lote_imagenes[i])
    plt.title(nombres_clase[int(lote_etiquetas[i])])
    plt.axis("off")
plt.show()

## 3. Define el aumento de datos y la CNN

In [ ]:
aumento_datos = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
])

modelo = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(128, 128, 3)),
    aumento_datos,
    tf.keras.layers.Conv2D(32, 3, activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(64, 3, activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(128, 3, activation="relu"),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])

modelo.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
modelo.summary()

In [ ]:
modelo.fit(dataset_entreno, epochs=5, validation_data=dataset_validacion)

## 4. Visualiza predicciones reales

In [ ]:
lote_imagenes, lote_etiquetas = next(iter(dataset_validacion))
predicciones = modelo.predict(lote_imagenes)

plt.figure(figsize=(12, 6))
for i in range(8):
    plt.subplot(2, 4, i + 1)
    plt.imshow(lote_imagenes[i])
    etiqueta_predicha = nombres_clase[int(predicciones[i] >= 0.5)]
    etiqueta_real = nombres_clase[int(lote_etiquetas[i])]
    plt.title(f"Real: {etiqueta_real}\nPredicho: {etiqueta_predicha}")
    plt.axis("off")
plt.tight_layout()
plt.show()

## Resumen

- Preprocesaste fotos reales de tamaño variable a un formato uniforme.
- El aumento de datos generó variaciones de cada foto para reducir el sobreajuste.
- Visualizar predicciones concretas revela qué tipo de errores comete el modelo, más allá de la exactitud global.

Continúa con el [notebook de práctica de esta clase](../notebooks_practica/clase08_practica_dogs_vs_cats.ipynb).